# Clayton Metang — Expedition
High-level workflow using `expedition.py`. Each cell calls one stage; `x.save()` persists config after each step.

In [1]:
%load_ext autoreload
%autoreload 2
import logging
import claytonlib as clayton
from claytonlib.expedition import expedition

# --- Logging ---
# INFO shows per-write-cycle timing; DEBUG adds per-turn RNG details
logging.basicConfig(level=logging.INFO)
# logging.getLogger('claytonlib').setLevel(logging.INFO)

x = expedition("metang")
x.reload()
x.print()
x.chart_options.evaluation_frames_per_write_cycle = 5

[expedition] Reloaded from data/expeditions/metang.json
=== Expedition: metang ===
  pokemon                  metang
  key_seed                 0x0C0E02C2
  setup_delay_s            180
  max_target_s             600
  strategy                 six-bait-then-balls
  criteria                 machete-50-turns-after-5-balls
  eval_strategy            sliding_window_13
  window                   120
  target_delay             20045
  initial_time             2000-07-24T14:45:55
  target_seeds             ['0x1C1562D2']
  metronome_histsz         10
  metronome_second_window  2
  compass_m_delay          2441
  ---
  delay_from_key           19339 frames  (322.32s)


In [16]:
x.adjust(strategy_name="six-bait-then-balls")

[expedition] strategy_name = 'six-bait-then-balls'


# Chart - Finding a target

## Create chart

In [2]:
x.precompute_chart()
x.save()

[expedition] 08:45:32  === precompute_chart ===  (2026-09-10)
[expedition] 08:45:32  charting metang key_seed=0x0C0E02C2 delay=180-600s strategy=six-bait-then-balls criteria=machete-50-turns-after-5-balls  workers=12
[expedition] 08:45:48  mdmsh 1/248  elapsed 0.3m  eta ~65.1m
[expedition] 08:46:46  mdmsh 5/248  elapsed 1.2m  eta ~60.1m
[expedition] 08:47:58  mdmsh 10/248  elapsed 2.4m  eta ~58.0m
[expedition] 08:49:09  mdmsh 15/248  elapsed 3.6m  eta ~56.1m
[expedition] 08:50:18  mdmsh 20/248  elapsed 4.8m  eta ~54.3m
[expedition] 08:51:31  mdmsh 25/248  elapsed 6.0m  eta ~53.3m
[expedition] 08:52:43  mdmsh 30/248  elapsed 7.2m  eta ~52.2m
[expedition] 08:53:52  mdmsh 35/248  elapsed 8.3m  eta ~50.6m
[expedition] 08:54:43  mdmsh 40/248  elapsed 9.2m  eta ~47.7m
[expedition] 08:55:47  mdmsh 45/248  elapsed 10.2m  eta ~46.2m
[expedition] 08:56:55  mdmsh 50/248  elapsed 11.4m  eta ~45.1m
[expedition] 08:58:07  mdmsh 55/248  elapsed 12.6m  eta ~44.1m
[expedition] 08:59:14  mdmsh 60/248  e

## Evaluate chart to find targets

`chart_report()` ranks the best **(boot time, commanded countdown M)** pairs across all candidate boot times (mode A), or the best M for a boot time you pass as `initial_time=` (mode B). It saves its ranked findings so `select_target()` can use them.

In [3]:
x.chart_report()
x.save()

[expedition] 09:51:32  === chart_report ===
Best (boot time, M) pairs  [top 10 of 1668]  (jitter kernel, k=3.5):
   #            boot time     M (ms)  target F_b  second  P(capture)   sigma
   1  2000-05-31 14:54:59     176148       10970     181       30.0%    51.7
   2  2000-05-30 14:59:59     244385       15062     250       29.4%    61.0
   3  2000-01-01 14:00:11     268131       16486     273       29.2%    63.8
   4  2000-06-26 14:53:59     327496       20046     333       28.9%    70.6
   5  2000-05-31 14:54:59     221172       13670     226       28.7%    58.0
   6  2000-01-01 14:00:11     381192       23266     386       28.6%    76.1
   7  2000-05-30 14:59:59     402136       24522     407       28.6%    78.2
   8  2000-05-30 14:59:59     411808       25102     417       28.5%    79.1
   9  2000-07-29 14:55:10     463236       28186     469       28.3%    83.9
  10  2000-05-31 14:54:59     179550       11174     185       28.3%    52.2
[expedition] 09:51:37  best target for e

## Choose Target

`select_target()` reads the findings `chart_report()` saved and lets you pick one. It records the chosen **boot time** (`initial_time`), **timer countdown** (`target_timer_delay` = M), and **expected battle frame** (`target_delay` = F_b) on the expedition, then saves.

In [5]:
x.select_target()

[expedition] 09:52:19  === select_target ===



Select by  [t] top ranking   [s] specific starting time  (blank to cancel):  s


Enter a starting time (year ignored). e.g. '2000-05-30 14:59:59' or '05-30 14:59:59'.


Starting time (blank to cancel):  2025-07-24 14:45:55


  -> best target for 2000-07-24 14:45:55: M=327496 ms, F_b=20046, P~28.9%
[expedition] Saved to data/expeditions/metang.json
[expedition] 09:52:30  target set: boot 2000-07-24T14:45:55, timer M=327496 ms, expected F_b=20046 (P~28.9%). Saved.
[expedition] 09:52:30  predicted battle time (m/d h:m:s): 07-24 14:51:28  (= boot + 333s; year is the chart's 2000)


{'rank': 1187,
 'initial_time': '2000-07-24T14:45:55',
 'M': 327496,
 'target_delay': 20046,
 'second': 333,
 'p': 0.28858095843370196,
 'sigma': 70.5578288636451,
 'mdmsh': [247, 14]}

## Examine target area

In [6]:
 x.check().chart_check_target_landing()


chart_check_target_landing  (jitter kernel, k=3.5)
boot=2000-07-24T14:45:55  timer M=327496 ms  ->  mean F_b=20046.0 (target_delay=20046)  sigma=70.6
target second=333  mdmsh(m,h)=(247, 14)  window frames [19799, 20293] (495)
predicted battle time (m/d h:m:s): 07-24 14:51:28  (seeds below are the ACTUAL hit seeds -- year folded into the frame via base_delay)
  frame      Δ        seed  hit    weight        w%      cumP%
--------------------------------------------------------------
  19799   -247  0xF70E4D57    ✓    0.0022    0.001%     0.001%
  19800   -246  0xF70E4D58    ✗    0.0023    0.001%     0.001%
  19801   -245  0xF70E4D59    ✗    0.0024    0.001%     0.001%
  19802   -244  0xF70E4D5A    ✓    0.0025    0.001%     0.003%
  19803   -243  0xF70E4D5B    ✗    0.0027    0.002%     0.003%
  19804   -242  0xF70E4D5C    ✗    0.0028    0.002%     0.003%
  19805   -241  0xF70E4D5D    ✗    0.0029    0.002%     0.003%
  19806   -240  0xF70E4D5E    ✓    0.0031    0.002%     0.004%
  19807 

{'p': 0.2885810571650803,
 'n_frames': 495,
 'n_captured': 120,
 'mismatches': None}

# Compass - Identify target

## Calibrate using metronome

In [ ]:
x.metronome_compass()
x.save()

## Finding what seed you hit in safari

`compass_safari()` builds candidates from the calibrated model: for the commanded countdown **M** (set by `select_target`) it sweeps the battle-frame window **F\* ± kσ** across second offsets **δ∈{−1,0,+1}** (off-by-one timer-start timing — each δ uses the *same* frame window). No hand-set delay window.

As you enter observed turns it ranks survivors by **posterior landing probability** (`P(land)`), shows the most-likely seed and which **δ** you hit ("timer on time / +1s late"), and flags when one candidate passes the confidence threshold. Extra commands:

- **`w`** — widen the frame (`k`) and/or second (`±K`) window and re-apply your path so far (also offered automatically on a no-match).
- The set is bounded to the seeds carrying `mass_cap` (default 0.999) of the landing probability; the Jane offload tip triggers on the *prior-weighted* effective count.

Pass `second_offsets=` / `mass_cap=` to override. Afterwards, `x.save_safari_run()` logs the identified seed, observed path, and inferred timer offset to `data/safari_runs.jsonl` (no capture required) for future model retuning.

In [ ]:
x.compass_safari()
x.save()

In [ ]:
# Loop-back: log this run (seed, observed path, inferred timer offset) for model retuning.
# No capture required — records even a fled/ambiguous run.
x.save_safari_run()

# Machete - Finding a path through seed

This is usually triggered during the "Finding what seed you hit in safari" step, but here's some manual activation anyways

## Finding a path for a single seed

In [ ]:
x.machete_one(max_turns=1000)
x.save()